# WC2026 — Exploratory Data Analysis (Track A)

Starter notebook for the first blog post. It loads our processed tables and seeds a few
**candidate angles** to explore — it deliberately draws **no conclusions**. Pick a thread,
dig in section by section, and take over in the *Scratch* section at the bottom.

> **Plotting needs matplotlib**, which lives in the Track B `viz` extra. Launch this notebook
> with the extra installed:
> ```bash
> uv sync --extra viz
> uv run --extra viz jupyter lab   # or open in VS Code using the .venv kernel
> ```

Data provenance: results/goals from **openfootball** (public domain); xG, team match stats,
venues, referees from the **Kaggle FIFA World Cup 2026** dataset (CC0, by MD Mominul Islam).
The join key throughout is our `match_id`.

## 0. Setup

In [ ]:
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fsc.utils import paths

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)


def load(name: str) -> pd.DataFrame:
    """Read a processed parquet table by its stem name (no hardcoded paths)."""
    return pd.read_parquet(paths.PROCESSED / f"{name}.parquet")


matches = load("matches")
goals = load("goals")
team_stats = load("team_match_stats")
venues = load("venues")
referees = load("referees")
crosswalk = load("match_crosswalk")

tables = {
    "matches": matches,
    "goals": goals,
    "team_match_stats": team_stats,
    "venues": venues,
    "referees": referees,
    "match_crosswalk": crosswalk,
}

In [ ]:
# Orient: shape, head, and dtypes for every table.
for name, df in tables.items():
    print(f"=== {name}: {df.shape[0]} rows x {df.shape[1]} cols ===")
    display(df.head())
    print(df.dtypes.to_string())
    print()

## 1. Data health / sanity

Quick orientation only — is the data shaped the way we expect before we lean on it?

In [ ]:
pd.set_option("display.max_rows", None)  # Show all rows in the next display.

In [ ]:
goals

In [ ]:
matches

In [ ]:
# Played vs scheduled, and the distinct round labels (group vs knockout structure).
print("status counts:")
print(matches["status"].value_counts(), "\n")
print("distinct rounds:")
for r in matches["round"].dropna().unique():
    print("  ", r)

In [ ]:
# Goals per (played) match: distribution + a quick histogram.
played_ids = set(matches.loc[matches["status"] == "played", "match_id"])
goals_per_match = (
    goals.groupby("match_id").size().reindex(list(played_ids), fill_value=0)
)
print(goals_per_match.describe())

ax = goals_per_match.plot(kind="hist", bins=range(0, goals_per_match.max() + 2), rwidth=0.9)
ax.set_xlabel("goals in match")
ax.set_title("Goals per played match")
plt.show()

In [ ]:
goals_per_match

In [ ]:
# Null check on key columns, and confirm team_match_stats has exactly 2 rows per played match.
key_cols = {
    "matches": ["match_id", "date", "team1", "team2", "status"],
    "team_match_stats": ["match_id", "team", "side", "xg"],
    "venues": ["name", "city", "elevation_m"],
}
for name, cols in key_cols.items():
    print(f"{name} nulls:")
    print(tables[name][cols].isna().sum().to_string(), "\n")

per_match = team_stats.groupby("match_id").size()
print("team_match_stats: matches covered =", per_match.size, "| all have 2 rows?", bool((per_match == 2).all()))
print("played matches =", len(played_ids))

## 2. Angle A — xG justice table (over/under-performance)

**Question:** which teams have scored more (or fewer) goals than the quality of their chances
suggests? **Columns:** `team_match_stats.xg` (chance quality) vs actual goals counted from
`goals`. Aggregated across all of a team's played matches.

> **Caveat:** this dataset's xG is *directionally* reliable but not identical to other providers'
> models. Read it as "chance quality", not a canonical xG number.

In [ ]:
actual_goals = goals.groupby("team").size().rename("goals")
team_xg = team_stats.groupby("team")["xg"].sum().rename("xg")

justice = pd.concat([team_xg, actual_goals], axis=1).fillna(0.0)
justice["diff"] = justice["goals"] - justice["xg"]
justice = justice.sort_values("diff")
display(justice)

In [ ]:
# Bar chart of (goals - xG): positive = out-performed chance quality, negative = under.
fig, ax = plt.subplots(figsize=(7, 9))
colors = ["#2a9d8f" if d >= 0 else "#e76f51" for d in justice["diff"]]
ax.barh(justice.index, justice["diff"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("goals − xG  (over / under performance)")
ax.set_title("xG justice table (chance quality vs goals scored)")
plt.tight_layout()
plt.show()

**Explore further:**
- Is over-performance driven by one hot match or spread across the tournament?
- Does it hold up per-90 / per-match rather than in aggregate?
- Split by stage — is finishing different in knockout vs group games?

In [ ]:
# One row per (match, team): xg (chance quality) vs goals actually scored, tagged by stage.
# This is the per-match decomposition of the section-2 aggregate justice table.
mt = team_stats[["match_id", "team", "xg"]].copy()
goals_per = goals.groupby(["match_id", "team"]).size().rename("goals")
mt = mt.merge(goals_per, on=["match_id", "team"], how="left")
mt["goals"] = mt["goals"].fillna(0).astype(int)
mt["diff"] = mt["goals"] - mt["xg"]                       # + = beat the chance quality
mt = mt.merge(matches[["match_id", "stage", "round", "date"]], on="match_id", how="left")

# Sanity: per-match goals sum back to the tournament total used in section 2.
assert mt["goals"].sum() == len(goals), "goal totals should reconcile with section 2"
display(mt.sort_values("diff", ascending=False).head())


In [ ]:
# For each team: total over/under-performance, plus how much of it comes from its single
# best match. best_share near 1.0 = one hot game carrying the number; near 1/n = evenly spread.
conc = mt.groupby("team").agg(
    total_diff=("diff", "sum"),
    best_match=("diff", "max"),
    n=("diff", "size"),
).sort_values("total_diff", ascending=False)
conc["best_share"] = (conc["best_match"] / conc["total_diff"]).where(conc["total_diff"] > 0)
display(conc.round(2))

# Strip plot: every match's diff for the biggest over/under-performers, so a lone spike is obvious.
top = conc["total_diff"].abs().sort_values(ascending=False).head(10).index[::-1]
fig, ax = plt.subplots(figsize=(8, 6))
for y, team in enumerate(top):
    d = mt.loc[mt["team"] == team, "diff"]
    ax.scatter(d, np.full(len(d), y), alpha=0.7,
               color=["#2a9d8f" if v >= 0 else "#e76f51" for v in d])
    ax.scatter(d.sum(), y, marker="|", s=400, color="black")  # team total
ax.axvline(0, color="black", linewidth=0.8)
ax.set_yticks(range(len(top))); ax.set_yticklabels(top)
ax.set_xlabel("goals − xG per match  (black bar = team total)")
ax.set_title("Is over-performance one hot match or spread out?")
plt.tight_layout(); plt.show()


In [ ]:
# Aggregate diff rewards teams that simply played more games (knockout runs). Normalise by
# matches played. (Each match ≈ 90', so diff-per-match is effectively the per-90 rate here.)
per90 = mt.groupby("team").agg(xg=("xg", "sum"), goals=("goals", "sum"), mp=("diff", "size"))
per90["diff"] = per90["goals"] - per90["xg"]
per90["diff_per_match"] = per90["diff"] / per90["mp"]
per90 = per90.sort_values("diff_per_match")

fig, ax = plt.subplots(figsize=(7, 9))
colors = ["#2a9d8f" if d >= 0 else "#e76f51" for d in per90["diff_per_match"]]
ax.barh(per90.index, per90["diff_per_match"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("(goals − xG) per match played")
ax.set_title("xG justice, normalised per match (does the rank order change?)")
plt.tight_layout(); plt.show()

# See who the normalisation reshuffles: aggregate rank vs per-match rank.
cmp = per90.assign(
    rank_agg=per90["diff"].rank(ascending=False),
    rank_per_match=per90["diff_per_match"].rank(ascending=False),
)
cmp["moved"] = cmp["rank_agg"] - cmp["rank_per_match"]
display(cmp.sort_values("moved", key=abs, ascending=False)[["diff", "diff_per_match", "moved"]].round(2))


In [ ]:
# Tournament-wide finishing ratio (goals / xg) by stage: >1 = teams beating their chances.
by_stage = mt.groupby("stage").agg(xg=("xg", "sum"), goals=("goals", "sum"), matches=("diff", "size"))
by_stage["finish_ratio"] = by_stage["goals"] / by_stage["xg"]
by_stage["diff_per_team_match"] = (by_stage["goals"] - by_stage["xg"]) / by_stage["matches"]
display(by_stage.round(3))

# Per-team finishing split, group vs knockout — only teams that reached the knockouts.
split = (mt.groupby(["team", "stage"])["diff"].sum().unstack(fill_value=np.nan))
split = split.dropna(subset=["knockout"]).sort_values("group")
x = np.arange(len(split)); w = 0.4
fig, ax = plt.subplots(figsize=(9, 6))
ax.bar(x - w/2, split["group"],    w, label="group",    color="#264653")
ax.bar(x + w/2, split["knockout"], w, label="knockout", color="#e9c46a")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x); ax.set_xticklabels(split.index, rotation=60, ha="right")
ax.set_ylabel("goals − xG"); ax.set_title("Finishing: group vs knockout (knockout teams only)")
ax.legend(); plt.tight_layout(); plt.show()


## 3. Angle B — the altitude angle

**Question:** does scoring change with stadium elevation? Mexico City (~2,200 m) and Guadalajara
sit far above the mostly sea-level US/Canada venues. **Columns:** `matches` (scores, venue) joined
to `venues.elevation_m`; goal minutes from `goals`.

> **Small-sample caution:** only a handful of matches are played at real altitude — treat any
> pattern as a prompt to look closer, not evidence.

In [ ]:
# matches.venue is a metro label ("Boston (Foxborough)"); venues.city is the suburb
# ("Foxborough"). Take the parenthetical when present, else the whole label -> venues.city.
def venue_city(v):
    if pd.isna(v):
        return None
    m = re.search(r"\((.*)\)", v)
    return m.group(1) if m else v


mv = matches[matches["status"] == "played"].copy()
mv["city"] = mv["venue"].map(venue_city)
mv = mv.merge(venues[["city", "elevation_m", "name"]], on="city", how="left")
mv["total_goals"] = mv["ft1"].astype("Float64") + mv["ft2"].astype("Float64")
print("venue->elevation join coverage:", mv["elevation_m"].notna().sum(), "/", len(mv))

mv["elev_bucket"] = pd.cut(
    mv["elevation_m"].astype("float"),
    bins=[-1, 100, 1000, 3000],
    labels=["sea level (<100m)", "mid (100–1000m)", "high (>1000m)"],
)
display(mv.groupby("elev_bucket", observed=True)["total_goals"].agg(["count", "mean"]))

In [ ]:
# Total shots per match by elevation bucket (shots summed across both teams).
shots_per_match = team_stats.groupby("match_id")["shots"].sum().rename("shots")
mv = mv.merge(shots_per_match, on="match_id", how="left")
display(mv.groupby("elev_bucket", observed=True)["shots"].agg(["count", "mean"]))

# First- vs second-half goal share by elevation (does scoring fade at altitude?).
gh = goals.merge(mv[["match_id", "elev_bucket"]], on="match_id", how="inner")
gh = gh[gh["minute"].notna()].copy()
gh["half"] = np.where(gh["minute"].astype("float") <= 45, "1st half", "2nd half")
half_share = (
    gh.groupby(["elev_bucket", "half"], observed=True).size().unstack(fill_value=0)
)
display(half_share.div(half_share.sum(axis=1), axis=0).round(3))

In [ ]:
# Scatter: elevation vs goals in the match.
fig, ax = plt.subplots(figsize=(8, 5))
d = mv.dropna(subset=["elevation_m", "total_goals"])
ax.scatter(d["elevation_m"].astype(float), d["total_goals"].astype(float), alpha=0.5)
ax.set_xlabel("stadium elevation (m)")
ax.set_ylabel("total goals in match")
ax.set_title("Elevation vs goals per match")
plt.show()

**Explore further:**
- Weight by number of matches — is the "high" bucket just one or two games?
- Do *shots* (not just goals) rise or fall with elevation, hinting at fatigue vs finishing?
- Is any half-time effect confounded by which teams happened to play up high?

In [ ]:
# Sample sizes behind each bucket mean — a mean over 9 matches at 2 venues is fragile.
sizes = mv.groupby("elev_bucket", observed=True).agg(
    matches=("match_id", "size"),
    venues=("name", "nunique"),
    cities=("city", "nunique"),
    mean_goals=("total_goals", "mean"),
)
sizes["mean_goals"] = sizes["mean_goals"].round(2)
display(sizes)

# List every high-altitude match individually — small enough to eyeball, so do.
high = mv[mv["elev_bucket"] == "high (>1000m)"]
display(high[["date", "team1", "team2", "name", "elevation_m", "total_goals", "shots"]]
        .sort_values("date").reset_index(drop=True))


In [ ]:
# Goals can swing on a couple of finishes; shots are a steadier proxy for how the game was
# played. If shots fall with altitude too, that hints at fatigue/tempo rather than finishing.
by_bucket = mv.groupby("elev_bucket", observed=True)["shots"].agg(["count", "mean"]).round(2)
display(by_bucket)

d = mv.dropna(subset=["elevation_m", "shots"])
r = d["elevation_m"].astype(float).corr(d["shots"].astype(float))
print(f"correlation elevation ~ total shots: {r:+.3f}  (n={len(d)})")

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(d["elevation_m"].astype(float), d["shots"].astype(float), alpha=0.5, color="#2a9d8f")
ax.set_xlabel("stadium elevation (m)")
ax.set_ylabel("total shots in match")
ax.set_title(f"Elevation vs shots per match  (r = {r:+.2f})")
plt.show()


In [ ]:
# The altitude sample isn't a random draw of teams: the host plays most of its games there.
# Count appearances at >1000m — if one team dominates, "altitude" and "that team" are tangled.
hi = mv[mv["elevation_m"].astype(float) > 1000]
appearances = (pd.concat([hi["team1"], hi["team2"]])
               .value_counts().rename("matches_at_altitude"))
display(appearances.to_frame())
print(f"{appearances.index[0]} accounts for {appearances.iloc[0]} of {len(hi)} high-altitude matches.")

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(appearances.index[::-1], appearances.values[::-1], color="#e76f51")
ax.set_xlabel("matches played at >1000 m")
ax.set_title("Who actually played at altitude? (confound check)")
plt.tight_layout(); plt.show()


## 4. Angle C — referee card patterns

**Question:** how much do referees differ in cards awarded? **Columns:** `referees.avg_cards_per_game`.

> **Note:** the processed tables don't yet carry a per-match referee assignment — that link lives
> in the raw Kaggle `matches_detailed.csv` (`referee_name`) / `matches.csv` (`referee_id`). So this
> is a distribution only; wiring the assignment into a processed table is a follow-up if the angle
> proves interesting.

In [ ]:
print(referees["avg_cards_per_game"].describe(), "\n")
display(referees.sort_values("avg_cards_per_game", ascending=False).head(10))

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(referees["avg_cards_per_game"], bins=12, rwidth=0.9)
ax.set_xlabel("avg cards per game")
ax.set_ylabel("referees")
ax.set_title("Distribution of referee card averages")
plt.show()

**Explore further:**
- Join `referee_name` from raw `matches_detailed.csv` to link refs to actual matches.
- Do high-card refs correlate with high-`fouls` matches in `team_match_stats`?
- Are strict refs concentrated in a particular confederation?

In [ ]:
# Referee assignment isn't in the processed tables yet — pull it from raw Kaggle.
# Join on referee_id (the reliable key): a few referee_name spellings in matches_detailed
# don't match the referees table, but every referee_id resolves.
RAW = paths.RAW / "kaggle_wc2026"
m_raw = pd.read_csv(RAW / "matches.csv")           # kaggle match_id -> referee_id
refs_raw = pd.read_csv(RAW / "referees.csv")        # referee_id -> name, country, avg_cards_per_game

ref_assign = m_raw[["match_id", "referee_id"]].merge(refs_raw, on="referee_id", how="left")
ref_assign = ref_assign.rename(columns={"match_id": "kaggle_match_id"})
print(f"assignments: {ref_assign['name'].notna().sum()} / {len(ref_assign)} matches have a named referee")

# Matches officiated per referee, alongside their reference card average.
by_ref = (ref_assign.groupby(["name", "country", "avg_cards_per_game"])
          .size().rename("matches").reset_index()
          .sort_values("avg_cards_per_game", ascending=False))
display(by_ref)


In [ ]:
# team_match_stats has no card column, so use fouls as the on-pitch proxy for a scrappy game.
# Question: do referees with a high career card average actually officiate higher-foul matches?
fouls_per_match = team_stats.groupby("kaggle_match_id")["fouls"].sum().rename("match_fouls")
rf = ref_assign.merge(fouls_per_match, on="kaggle_match_id", how="inner").dropna(
    subset=["avg_cards_per_game", "match_fouls"])

r = rf["avg_cards_per_game"].corr(rf["match_fouls"])
print(f"correlation  ref avg_cards ~ match fouls:  {r:+.3f}   (n={len(rf)} matches)")

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(rf["avg_cards_per_game"], rf["match_fouls"], alpha=0.5, color="#2a9d8f")
ax.set_xlabel("referee's career avg cards per game")
ax.set_ylabel("total fouls in the match")
ax.set_title(f"Do strict refs get scrappier games?  (r = {r:+.2f})")
plt.show()

# Per-referee: does a high-card ref's mean match-fouls actually run high?
per_ref = rf.groupby("name").agg(
    avg_cards=("avg_cards_per_game", "first"),
    mean_match_fouls=("match_fouls", "mean"),
    matches=("match_fouls", "size"),
).sort_values("avg_cards", ascending=False)
display(per_ref.round(2))


In [ ]:
# No confederation column in the data — map referee nationality to confederation by hand.
# (Every referee country in this dataset resolves; adjust the map if new refs appear.)
CONFED = {
    "UEFA": {"Poland", "Italy", "England", "France", "Netherlands", "Norway",
             "Portugal", "Slovenia", "Sweden"},
    "CONMEBOL": {"Argentina", "Brazil", "Uruguay", "Venezuela"},
    "CONCACAF": {"Canada", "Mexico", "El Salvador", "Honduras", "USA", "United States"},
    "CAF": {"Algeria", "Morocco", "South Africa"},
    "AFC": {"Australia", "China", "Japan", "Jordan"},
}
country_to_confed = {c: k for k, cs in CONFED.items() for c in cs}
refs_raw["confed"] = refs_raw["country"].map(country_to_confed)
missing = set(refs_raw.loc[refs_raw["confed"].isna(), "country"])
if missing:
    print("⚠ unmapped referee countries (add them to CONFED):", missing)

by_confed = (refs_raw.groupby("confed")["avg_cards_per_game"]
             .agg(["count", "mean", "min", "max"]).round(2).sort_values("mean"))
display(by_confed)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(by_confed.index, by_confed["mean"], color="#e76f51")
ax.set_ylabel("mean avg cards per game")
ax.set_title("Referee strictness by confederation")
for i, (n, m) in enumerate(zip(by_confed["count"], by_confed["mean"])):
    ax.text(i, m, f"n={n}", ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.show()


## 5. Angle D — efficiency: possession vs xG vs result

**Question:** does more of the ball translate into better chances, and into winning? **Columns:**
`team_match_stats.possession`, `.xg`, joined to each team's match result from `matches`.

> **Caveat:** result here is from the **full-time (90')** score, so a knockout tie decided in extra
> time or on penalties shows up as a *draw*.

In [ ]:
outcomes = matches.loc[
    matches["status"] == "played", ["match_id", "team1", "team2", "ft1", "ft2"]
]
ts = team_stats.merge(outcomes, on="match_id", how="inner")


def result(row):
    if row["team"] == row["team1"]:
        gf, ga = row["ft1"], row["ft2"]
    elif row["team"] == row["team2"]:
        gf, ga = row["ft2"], row["ft1"]
    else:
        return None
    if pd.isna(gf) or pd.isna(ga):
        return None
    return "win" if gf > ga else ("draw" if gf == ga else "loss")


ts["result"] = ts.apply(result, axis=1)
print("rows without a resolved result:", int(ts["result"].isna().sum()), "of", len(ts))
display(ts.groupby("result")[["possession", "xg"]].mean())

In [ ]:
# Scatter: possession vs xG, colored by result.
fig, ax = plt.subplots(figsize=(8, 5))
palette = {"win": "#2a9d8f", "draw": "#e9c46a", "loss": "#e76f51"}
for res, sub in ts.dropna(subset=["result", "possession", "xg"]).groupby("result"):
    ax.scatter(
        sub["possession"].astype(float), sub["xg"].astype(float),
        label=res, alpha=0.6, color=palette.get(res),
    )
ax.set_xlabel("possession %")
ax.set_ylabel("xG")
ax.set_title("Possession vs xG, by full-time result")
ax.legend()
plt.show()

**Explore further:**
- Bucket possession (e.g. <40 / 40–60 / >60) and look at win rate per bucket.
- Are high-possession-low-xG teams a real "sterile domination" cluster?
- Swap xG for shots-on-target — does the story change?

In [ ]:
# Does more of the ball actually convert to points? Bucket possession and read win rate + PPG.
t = ts.dropna(subset=["result", "possession"]).copy()
t["poss_bucket"] = pd.cut(t["possession"].astype(float), bins=[0, 40, 60, 100],
                          labels=["<40%", "40–60%", ">60%"])
t["points"] = t["result"].map({"win": 3, "draw": 1, "loss": 0})
t["is_win"] = (t["result"] == "win").astype(int)

by_poss = t.groupby("poss_bucket", observed=True).agg(
    n=("result", "size"),
    win_rate=("is_win", "mean"),
    ppg=("points", "mean"),
).round(3)
display(by_poss)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(by_poss.index.astype(str), by_poss["win_rate"], color="#2a9d8f")
ax.set_ylabel("win rate"); ax.set_ylim(0, 1)
ax.set_title("Win rate by possession bucket")
for i, (nn, wr) in enumerate(zip(by_poss["n"], by_poss["win_rate"])):
    ax.text(i, wr, f"{wr:.0%}\n(n={nn})", ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.show()


In [ ]:
# Sterile domination = lots of the ball, few good chances. Flag high possession + below-median
# xG, then check whether that quadrant actually loses/draws (the "all possession, no bite" story).
med_xg = t["xg"].astype(float).median()
t["quadrant"] = np.where(
    (t["possession"].astype(float) > 60) & (t["xg"].astype(float) < med_xg),
    "sterile (poss>60, low xG)", "other")

sterile = t[t["quadrant"] == "sterile (poss>60, low xG)"]
print(f"median xG = {med_xg:.2f} | sterile cluster: {len(sterile)} team-matches")
display(sterile["result"].value_counts())
display(sterile[["team", "possession", "xg", "shots_on_target", "result"]]
        .sort_values("possession", ascending=False))

# Same scatter as section 5, but highlight the sterile quadrant.
fig, ax = plt.subplots(figsize=(8, 5))
oth = t[t["quadrant"] == "other"]
ax.scatter(oth["possession"].astype(float), oth["xg"].astype(float), alpha=0.35, color="#adb5bd", label="other")
ax.scatter(sterile["possession"].astype(float), sterile["xg"].astype(float),
           color="#e76f51", s=70, label="sterile domination")
ax.axhline(med_xg, ls="--", color="black", lw=0.8); ax.axvline(60, ls="--", color="black", lw=0.8)
ax.set_xlabel("possession %"); ax.set_ylabel("xG")
ax.set_title("Sterile domination quadrant"); ax.legend()
plt.show()


In [ ]:
# xG is model-flavoured; shots-on-target is a rawer "did you actually test the keeper" count.
# If possession→SoT tells the same story as possession→xG, the finding is robust to that swap.
print(f"corr  possession ~ xG              : {t['possession'].astype(float).corr(t['xg'].astype(float)):+.3f}")
print(f"corr  possession ~ shots_on_target : {t['possession'].astype(float).corr(t['shots_on_target'].astype(float)):+.3f}")
display(t.groupby("result")[["possession", "xg", "shots_on_target"]].mean().round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True)
palette = {"win": "#2a9d8f", "draw": "#e9c46a", "loss": "#e76f51"}
for ax, ycol in zip(axes, ["xg", "shots_on_target"]):
    for res, sub in t.dropna(subset=["result", ycol]).groupby("result"):
        ax.scatter(sub["possession"].astype(float), sub[ycol].astype(float),
                   label=res, alpha=0.6, color=palette.get(res))
    ax.set_xlabel("possession %"); ax.set_ylabel(ycol); ax.set_title(f"possession vs {ycol}")
axes[0].legend()
plt.tight_layout(); plt.show()


## 6. Scratch

Your turn — take over from here.